In [2]:
# =====================================================================
# Gate L, chunk L0 -- INSPECT the published Qwen3.5-4B J/R lens pair.
#   1. pin the working directory (server data folder)
#   2. lens repos still at the commits recorded in setup_spec.json, trees clean?
#   3. sha256 of both lens files (canonical lens IDs from here on)
#   4. print each lens's stored provenance (recipe, prompts, config_json)
#   5. matrix sanity: shapes, anchor row == I, J-vs-R per-layer similarity
#      (the J/R gap is the yardstick for how tight the L1 refit bar can be)
#   6. jlens.fit signature, and whether pile-10k is available locally
# CPU only, nothing fitted.  Writes lens_calibration/l0_published_lens_provenance.json
# =====================================================================
import os
from pathlib import Path
os.chdir("/mnt/c/Users/bfabe/workspace/euphoria-axis")
assert Path("runs/qwen35-4b-posttrained-s2/paper_protocol").is_dir(), Path.cwd()

import hashlib
import inspect
import json
import subprocess
import sys

import numpy as np
import torch

paper_run_dir = Path("runs/qwen35-4b-posttrained-s2/paper_protocol")
cal_dir = paper_run_dir / "lens_calibration"
cal_dir.mkdir(parents=True, exist_ok=True)
setup = json.loads((paper_run_dir / "lens_readout" / "setup_spec.json").read_text())

def git(path, *args):
    return subprocess.check_output(["git", "-C", str(path), *args], text=True).strip()

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()

# ---------------------------------------------------------------------
# 2. repos vs the commits recorded when the lens readouts were made
# ---------------------------------------------------------------------
repos = {"jacobian-lens": (Path(setup["jacobian_lens_repo"]), setup["jacobian_lens_commit"]),
         "workspace-lenses": (Path(setup["workspace_lenses_repo"]), setup["workspace_lenses_commit"])}
repo_state = {}
for name, (path, recorded) in repos.items():
    head = git(path, "rev-parse", "HEAD")
    dirty = git(path, "status", "--porcelain", "--untracked-files=no")
    repo_state[name] = {"path": str(path), "recorded": recorded, "head": head, "matches": head == recorded,
                        "clean": dirty == ""}
    print(f"{name:17s} HEAD {head[:12]}  recorded {recorded[:12]}  "
          f"{'MATCH' if head == recorded else 'DIFFERENT'}  {'clean' if not dirty else 'MODIFIED FILES:'}")
    if dirty:
        print(dirty)

# ---------------------------------------------------------------------
# 3-4. lens files: hash, keys, provenance
# ---------------------------------------------------------------------
lens_dir4b = Path(setup["workspace_lenses_repo"]) / "qwen3.5-4b"
print("\nqwen3.5-4b/ contents:", sorted(str(p.relative_to(lens_dir4b)) for p in lens_dir4b.rglob("*") if p.is_file()))
raw, info = {}, {}
for name in ["J", "R"]:
    p = Path(setup["lens_files"][name])
    raw[name] = torch.load(p, map_location="cpu", weights_only=False)
    d = raw[name]
    prov = d.get("provenance", {}) if isinstance(d, dict) else {}
    cfg = prov.get("config_json")
    if isinstance(cfg, str):
        try:
            cfg = json.loads(cfg)
        except json.JSONDecodeError:
            pass
    info[name] = {"path": str(p), "bytes": p.stat().st_size, "sha256": sha256(p),
                  "keys": sorted(d.keys()) if isinstance(d, dict) else str(type(d)),
                  "n_prompts": d.get("n_prompts") if isinstance(d, dict) else None,
                  "d_model": d.get("d_model") if isinstance(d, dict) else None,
                  "source_layers": list(d.get("source_layers", [])) if isinstance(d, dict) else None,
                  "provenance": {k: v for k, v in prov.items() if k != "config_json"}, "config": cfg}
    print(f"\n===== {name} lens =====")
    print(f"sha256 {info[name]['sha256']}  ({info[name]['bytes'] / 2**20:.1f} MiB)")
    print("keys:", info[name]["keys"], "| n_prompts:", info[name]["n_prompts"], "| d_model:", info[name]["d_model"])
    print("provenance:", json.dumps(info[name]["provenance"], indent=1, default=str)[:2500])
    print("config_json:", json.dumps(cfg, indent=1, default=str)[:4000] if cfg is not None else "(none)")

# ---------------------------------------------------------------------
# 5. matrix sanity and the J-vs-R gap
# ---------------------------------------------------------------------
def stack(d):
    J = d["J"]
    return J.float() if torch.is_tensor(J) else torch.stack([J[k].float() for k in sorted(J)])
JJ, RR = stack(raw["J"]), stack(raw["R"])
print(f"\nJ stack {tuple(JJ.shape)} {raw['J']['J'].dtype if torch.is_tensor(raw['J']['J']) else 'dict'} | "
      f"R stack {tuple(RR.shape)}")
eye = torch.eye(JJ.shape[-1])
print("last row == I  (J, R):", bool(torch.allclose(JJ[-1], eye, atol=1e-3)), bool(torch.allclose(RR[-1], eye, atol=1e-3)))
rows = []
for i in range(JJ.shape[0]):
    a, b = JJ[i].flatten().double(), RR[i].flatten().double()
    rows.append({"row": i, "J_fro": float(a.norm()), "R_fro": float(b.norm()),
                 "cos_J_R": float(a @ b / (a.norm() * b.norm()))})
print("per-layer Frobenius cosine J vs R (row: cos):",
      " ".join(f"{r['row']}:{r['cos_J_R']:.3f}" for r in rows))

# ---------------------------------------------------------------------
# 6. fitting entry point and data availability
# ---------------------------------------------------------------------
sys.path.insert(0, setup["jacobian_lens_repo"])
import jlens
print("\njlens.fit", inspect.signature(jlens.fit))
print((inspect.getdoc(jlens.fit) or "")[:1500])
pile = None
try:
    from datasets import load_dataset
    pile = load_dataset("NeelNanda/pile-10k", split="train", download_mode="reuse_cache_if_exists")
    print(f"\npile-10k available: {len(pile)} docs; first doc starts {pile[0]['text'][:80]!r}")
except Exception as e:
    print(f"\npile-10k NOT available locally/online here: {type(e).__name__}: {str(e)[:200]}")

(cal_dir / "l0_published_lens_provenance.json").write_text(json.dumps(
    {"repos": repo_state, "lenses": {k: {kk: vv for kk, vv in v.items() if kk != "source_layers"} | {"source_layers": v["source_layers"]}
                                     for k, v in info.items()},
     "J_vs_R_per_layer": rows, "pile10k_available": pile is not None}, indent=2, default=str) + "\n")
print("\nsaved:", cal_dir / "l0_published_lens_provenance.json")

jacobian-lens     HEAD 581d398613e5  recorded 581d398613e5  MATCH  clean
workspace-lenses  HEAD d740106d1e0f  recorded d740106d1e0f  MATCH  clean

qwen3.5-4b/ contents: ['j-lens/lens.pt', 'r-lens/lens.pt']

===== J lens =====
sha256 8ac16d8e5e988f19d9e19ac9b459c9e73336a866c8534d521200e4fcf2c21377  (387.5 MiB)
keys: ['J', 'd_model', 'n_prompts', 'provenance', 'source_layers'] | n_prompts: 25 | d_model: 2560
provenance: {
 "model_id": "Qwen/Qwen3.5-4B",
 "dataset_id": "NeelNanda/pile-10k",
 "target_layer": 30,
 "t_max": 128,
 "n_prompts": 25,
 "docs_consumed": 25,
 "n_positions": 0.0,
 "git_commit": "modal",
 "skip_first": 4,
 "weighting": "uniform",
 "corpus_mode": "pretrain"
}
config_json: {
 "estimator": "standard"
}

===== R lens =====
sha256 848b9b8e76eb0448a926edc8bb374cb4369916dd6bdbd62f1d3141fa12b9f231  (387.5 MiB)
keys: ['J', 'd_model', 'n_prompts', 'provenance', 'source_layers'] | n_prompts: 25 | d_model: 2560
provenance: {
 "model_id": "Qwen/Qwen3.5-4B",
 "dataset_id": "NeelNa

/mnt/c/Users/bfabe/workspace/bookspace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████████████████████████████████████| 10000/10000 [00:00<00:00, 44574.50 examples/s]


pile-10k available: 10000 docs; first doc starts 'It is done, and submitted. You can play “Survival of the Tastiest” on Android, a'

saved: runs/qwen35-4b-posttrained-s2/paper_protocol/lens_calibration/l0_published_lens_provenance.json


In [1]:
print(1)

1


In [ ]:
# =====================================================================
# Gate L, chunk L1a -- refit the Qwen3.5-4B J lens with the published recipe
# (first 25 pile-10k docs, max_seq_len 128, skip_first 4, target layer 30,
# official jlens.fit) and compare it, layer by layer, with the published J.
# Run once with FORCE_BOS = True (jlens default) and, if that does not match,
# once with False.  Resumable via jlens checkpoints.
#
# Pass bar (L1, per layer l < 30):
#   1 - cos(refit_J, published_J)  <  1 - cos(published_J, published_R)
# i.e. the refit sits closer to its own published lens than J sits to R.
#
# REMINDER (Mac side): add to .gitignore before the next commit:
#   runs/**/lens_calibration/*.pt
# =====================================================================
import os
from pathlib import Path
os.chdir("/mnt/c/Users/bfabe/workspace/euphoria-axis")
assert Path("runs/qwen35-4b-posttrained-s2/paper_protocol").is_dir(), Path.cwd()

import hashlib
import json
import sys
import time

import torch
import transformers

FORCE_BOS = True                       # second run: False
N_DOCS, MAX_SEQ_LEN, SKIP_FIRST, TARGET = 25, 128, 4, 30
DIM_BATCH = 8

paper_run_dir = Path("runs/qwen35-4b-posttrained-s2/paper_protocol")
cal_dir = paper_run_dir / "lens_calibration"
setup = json.loads((paper_run_dir / "lens_readout" / "setup_spec.json").read_text())
l0 = json.loads((cal_dir / "l0_published_lens_provenance.json").read_text())
sys.path.insert(0, setup["jacobian_lens_repo"])
import jlens
from datasets import load_dataset

# ---------------------------------------------------------------------
# model (once per kernel) and the published recipe's prompts
# ---------------------------------------------------------------------
if "hf" not in globals():
    hf = transformers.AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3.5-4B", torch_dtype=torch.bfloat16, local_files_only=True).cuda().eval()
    tok = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", local_files_only=True)
fit_model = jlens.from_hf(hf, tok, force_bos=FORCE_BOS)
pile = load_dataset("NeelNanda/pile-10k", split="train")
prompts = [pile[i]["text"] for i in range(N_DOCS)]
prompts_sha = hashlib.sha256(json.dumps(prompts).encode()).hexdigest()
print(f"force_bos={FORCE_BOS} | {len(prompts)} prompts (sha {prompts_sha[:16]}) | first: {prompts[0][:60]!r}")

# ---------------------------------------------------------------------
# fit (resumable)
# ---------------------------------------------------------------------
tag = f"refit_J_n{N_DOCS}_bos{int(FORCE_BOS)}"
ckpt = cal_dir / f"{tag}.ckpt.pt"
out = cal_dir / f"{tag}.pt"
if not out.exists():
    t0 = time.time()
    lens = jlens.fit(fit_model, prompts, target_layer=TARGET, max_seq_len=MAX_SEQ_LEN, skip_first=SKIP_FIRST,
                     dim_batch=DIM_BATCH, checkpoint_path=str(ckpt), checkpoint_every=5, resume=True)
    lens.save(str(out))
    print(f"fit done in {(time.time() - t0) / 60:.1f} min -> {out}")
    if ckpt.exists():
        ckpt.unlink()
else:
    print("refit exists:", out)

# ---------------------------------------------------------------------
# compare with the published pair
# ---------------------------------------------------------------------
def stack(path):
    d = torch.load(path, map_location="cpu", weights_only=False)
    J = d["J"]
    return (J.float() if torch.is_tensor(J) else torch.stack([J[k].float() for k in sorted(J)])), d

pubJ, _ = stack(setup["lens_files"]["J"])
pubR, _ = stack(setup["lens_files"]["R"])
refJ, refd = stack(out)
print("refit keys:", sorted(refd.keys()), "| source_layers:", list(refd.get("source_layers", []))[:3], "...",
      list(refd.get("source_layers", []))[-3:], "| stack", tuple(refJ.shape))
n = min(refJ.shape[0], pubJ.shape[0], TARGET)
cos = lambda a, b: float((a.flatten().double() @ b.flatten().double()) / (a.double().norm() * b.double().norm()))
rows, passed = [], True
for l in range(n):
    c_ref, c_jr = cos(refJ[l], pubJ[l]), cos(pubJ[l], pubR[l])
    ok = (1 - c_ref) < (1 - c_jr)
    passed &= ok
    rows.append({"layer": l, "cos_refit_vs_pubJ": c_ref, "cos_pubJ_vs_pubR": c_jr,
                 "dist_ratio": (1 - c_ref) / max(1 - c_jr, 1e-12), "pass": ok})
print(f"\nlayer  cos(refit,pubJ)  cos(pubJ,pubR)  dist ratio (<1 passes)")
for r in rows:
    print(f"{r['layer']:5d}  {r['cos_refit_vs_pubJ']:15.5f}  {r['cos_pubJ_vs_pubR']:14.5f}  {r['dist_ratio']:9.3f}"
          f"  {'' if r['pass'] else 'FAIL'}")
print(f"\nL1a (force_bos={FORCE_BOS}): {'PASS' if passed else 'FAIL'} at {sum(r['pass'] for r in rows)}/{n} layers")
(cal_dir / f"l1a_{tag}.json").write_text(json.dumps(
    {"force_bos": FORCE_BOS, "recipe": {"n_docs": N_DOCS, "max_seq_len": MAX_SEQ_LEN, "skip_first": SKIP_FIRST,
                                        "target_layer": TARGET, "dim_batch": DIM_BATCH},
     "prompts_sha256": prompts_sha, "refit_sha256": hashlib.sha256(out.read_bytes()).hexdigest(),
     "published_J_sha256": l0["lenses"]["J"]["sha256"], "rows": rows, "pass": passed}, indent=2) + "\n")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████████████████████████████████████████████| 426/426 [00:00<00:00, 1957.39it/s]


force_bos=True | 25 prompts (sha 4111a9cf7ec45fdf) | first: 'It is done, and submitted. You can play “Survival of the Tas'


/mnt/c/Users/bfabe/workspace/bookspace/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:979: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:408.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
